<a href="https://colab.research.google.com/github/chaithu-gif/AI_LAB/blob/main/lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
class PuzzleState:
    def __init__(self, board, blank_row, blank_col, depth=0, path=None, board_history=None):
        self.board = board
        # Track coordinates of the blank space (represented by 0)
        self.blank_row = blank_row
        self.blank_col = blank_col
        self.depth = depth
        self.path = path if path is not None else []
        # Keep track of the full history of boards visited to reach this state
        self.board_history = board_history if board_history is not None else [board]

    # Make the board state hashable so we can keep track of visited configurations
    def get_tuple(self):
        return tuple(tuple(row) for row in self.board)

def print_board(board):
    for row in board:
        print(" ".join(map(str, row)))
    print("-" * 10)

def get_neighbors(state):
    neighbors = []
    # Possible moves: Up, Down, Left, Right
    moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

    for dr, dc, move_name in moves:
        new_row, new_col = state.blank_row + dr, state.blank_col + dc

        # Check boundary conditions
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            # Create a deep copy of the board to make the move
            new_board = [row[:] for row in state.board]

            # Swap the blank tile (0) with the adjacent tile
            new_board[state.blank_row][state.blank_col], new_board[new_row][new_col] = \
                new_board[new_row][new_col], new_board[state.blank_row][state.blank_col]

            neighbors.append(PuzzleState(
                new_board,
                new_row,
                new_col,
                state.depth + 1,
                state.path + [move_name],
                state.board_history + [new_board]  # Append the new board to history
            ))

    return neighbors

def solve_8_puzzle_dfs(start_board, goal_board, max_depth=15):
    # Find the starting coordinates of the blank tile (0)
    start_r, start_c = 0, 0
    for r in range(3):
        for c in range(3):
            if start_board[r][c] == 0:
                start_r, start_c = r, c

    # Initialize the starting state
    start_state = PuzzleState(start_board, start_r, start_c)
    goal_tuple = tuple(tuple(row) for row in goal_board)

    # DFS stack storing the PuzzleState objects
    stack = [start_state]

    # Visited set stores the map configuration paired with the depth it was visited at
    visited = {}

    while stack:
        curr_state = stack.pop()
        curr_tuple = curr_state.get_tuple()

        # Goal check
        if curr_tuple == goal_tuple:
            # Return the path, depth, AND the list of boards seen along the way
            return curr_state.path, curr_state.depth, curr_state.board_history

        # Depth bounding optimization
        if curr_state.depth >= max_depth:
            continue

        # Skip if we already visited this state at a smaller or equal depth
        if curr_tuple in visited and visited[curr_tuple] <= curr_state.depth:
            continue

        visited[curr_tuple] = curr_state.depth

        # Get legal moves and push them onto the stack
        for neighbor in get_neighbors(curr_state):
            stack.append(neighbor)

    return None, None, None

# --- Demonstration Execution ---
if __name__ == "__main__":
    # 0 represents the empty/blank space
    initial_state = [
        [1, 2, 3],
        [0, 4, 6],
        [7, 5, 8]
    ]

    target_state = [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 0]
    ]
    print("1WN24CS067 \n Chaithanya D K")
    print("Initial Board:")
    print_board(initial_state)

    # Solve the puzzle using DFS with a max depth cut-off of 10 moves
    moves_sequence, execution_depth, history = solve_8_puzzle_dfs(initial_state, target_state, max_depth=10)

    if moves_sequence:
        print(f" Goal reached successfully at depth: {execution_depth}!\n")

        print("--- Step-by-Step State Transition ---")
        for index, board in enumerate(history):
            if index == 0:
                print("Start State:")
            else:
                print(f"Step {index} (Moved {moves_sequence[index-1]}):")
            print_board(board)

        print(f"Path taken: {' -> '.join(moves_sequence)}")
    else:
        print(" Could not find a solution within the specified max depth limit.")

1WN24CS067 
 Chaithanya D K
Initial Board:
1 2 3
0 4 6
7 5 8
----------
 Goal reached successfully at depth: 3!

--- Step-by-Step State Transition ---
Start State:
1 2 3
0 4 6
7 5 8
----------
Step 1 (Moved Right):
1 2 3
4 0 6
7 5 8
----------
Step 2 (Moved Down):
1 2 3
4 5 6
7 0 8
----------
Step 3 (Moved Right):
1 2 3
4 5 6
7 8 0
----------
Path taken: Right -> Down -> Right
